<a href="https://colab.research.google.com/github/shagunsri09/Digital-product-funnel-and-retention-analytics-/blob/main/HclProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#admission no:- 2026b15310466
#name: Shagun Srivastava
#serial no: 39
#Github :- https://github.com/shagunsri09/Digital-product-funnel-and-retention-analytics-.git
from google.colab import files

In [2]:
upload = files.upload()
filename = next(iter(upload))

Saving project16_synthetic_ecommerce_events_1.csv to project16_synthetic_ecommerce_events_1.csv


In [3]:
import numpy as np
import pandas as pd


In [4]:
df = pd.read_csv(filename)

In [5]:
df.head()

,event_id,user_id,session_id,event_name,timestamp,device,acquisition_source,product_id
0,E00009753,U01825,U01825_S03,product_view,2026-02-13 22:39:18,Mobile,Instagram,P025
1,E00007466,U01389,U01389_S01,product_view,2026-03-17 07:28:29,Tablet,Google,P014
2,E00008623,U01608,U01608_S03,app_open,2026-03-25 13:09:00,Mobile,YouTube,NaN
3,E00001776,U00315,U00315_S02,app_open,2026-02-13 09:22:00,Mobile,Direct,NaN
4,E00003466,U00631,U00631_S03,product_view,2026-02-03 11:07:20,Mobile,Google,P004


In [6]:
#Event grain represents what one row in dataset represents.
#In this project each row is representing one individual user activity at specific timestamps.

#check deduplicate events.
df.duplicated(subset="event_id").sum()

np.int64(106)

In [7]:
df_clean = df.drop_duplicates(subset = "event_id")

In [8]:
df_clean["event_id"].duplicated().sum()

np.int64(0)

In [9]:
df_clean.head()

,event_id,user_id,session_id,event_name,timestamp,device,acquisition_source,product_id
0,E00009753,U01825,U01825_S03,product_view,2026-02-13 22:39:18,Mobile,Instagram,P025
1,E00007466,U01389,U01389_S01,product_view,2026-03-17 07:28:29,Tablet,Google,P014
2,E00008623,U01608,U01608_S03,app_open,2026-03-25 13:09:00,Mobile,YouTube,NaN
3,E00001776,U00315,U00315_S02,app_open,2026-02-13 09:22:00,Mobile,Direct,NaN
4,E00003466,U00631,U00631_S03,product_view,2026-02-03 11:07:20,Mobile,Google,P004


In [10]:
#convert timestamp into datetime format
df_clean["timestamp"] = pd.to_datetime(df_clean["timestamp"])

/tmp/ipykernel_1556/58408006.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean["timestamp"] = pd.to_datetime(df_clean["timestamp"])


In [11]:
#sort events by user, session and timestamp
df_sorted = df_clean.sort_values(by=["user_id","session_id","timestamp"])

In [12]:
df_sorted.head()

,event_id,user_id,session_id,event_name,timestamp,device,acquisition_source,product_id
1478,E00000001,U00001,U00001_S01,app_open,2026-01-09 19:39:00,Desktop,Direct,NaN
9185,E00000002,U00001,U00001_S01,product_view,2026-01-09 19:39:45,Desktop,Direct,P021
6344,E00000003,U00001,U00001_S01,add_to_cart,2026-01-09 19:41:29,Desktop,Direct,P021
516,E00000004,U00001,U00001_S02,app_open,2026-01-23 20:27:00,Desktop,Direct,NaN
10386,E00000005,U00001,U00001_S02,product_view,2026-01-23 20:27:55,Desktop,Direct,P004


In [13]:
#check user session mapping.
df_sorted.groupby("session_id")["user_id"].nunique().value_counts()

,count
user_id,
1,3969


In [14]:
#check first and last event of each session
df_sorted.groupby("session_id")["event_name"].agg(["first","last"]).value_counts()

first     last        
app_open  product_view    2520
          add_to_cart      628
          purchase         466
          checkout         355
Name: count, dtype: int64

In [15]:
#define expected event order
event_order = {
    "app_open": 1,
        "product_view": 2,
            "add_to_cart": 3,
                "checkout": 4,
                    "purchase": 5
                    }

In [16]:
#assign order number
df_sorted["event_order"] = df_sorted["event_name"].map(event_order)

In [17]:
from types import LambdaType
#validate event ordering within each session
df_sorted.groupby("session_id")["event_order"].apply(lambda x: x.is_monotonic_increasing).value_counts()

,count
event_order,
True,3969


In [18]:
# count unique users at each funnel  stage
funnel_counts = df_clean.groupby("event_name")["user_id"].nunique()

In [21]:
#arrange stages in funnel order
funnel_counts = funnel_counts.reindex([
    "app_open",
        "product_view",
            "add_to_care",
                "checkout",
                    "purchase"
                    ])


In [22]:
funnel_counts

,user_id
event_name,
app_open,2000.0
product_view,2000.0
add_to_care,NaN
checkout,724.0
purchase,434.0


In [23]:
#calculate conversion rates at each funnel stages
conversion_rate = (funnel_counts / funnel_counts.iloc[0]) * 100
conversion_rate

,user_id
event_name,
app_open,100.0
product_view,100.0
add_to_care,NaN
checkout,36.2
purchase,21.7


In [24]:
user_purchase = df_clean.groupby("user_id")["event_name"].apply(lambda x: int("purchase" in x.values))

In [27]:
#bootstrap confidence interval
bootstrap_rates = []
for i in range(1000):
  sample = np.random.choice(user_purchase.values, size=len(user_purchase),replace=True)




In [30]:
np.percentile(bootstrap_rates , [2.5, 97.5])

array([20.65, 20.65])

In [31]:
device_purchase = df_clean.groupby("device")["event_name"].apply(lambda x: (x=="purchase").sum())

In [32]:
device_purchase

,event_name
device,
Desktop,100
Mobile,342
Tablet,24


In [38]:
df_cohort = df_clean.copy()
df_cohort["activity_month"] = df_cohort["timestamp"].dt.to_period("M")

In [39]:
df_cohort["cohort_month"] = df_cohort.groupby("user_id")["timestamp"].transform("min").dt.to_period("M")

In [40]:
df_cohort.head()

,event_id,user_id,session_id,event_name,timestamp,device,acquisition_source,product_id,activity_month,cohort_month
0,E00009753,U01825,U01825_S03,product_view,2026-02-13 22:39:18,Mobile,Instagram,P025,2026-02,2026-02
1,E00007466,U01389,U01389_S01,product_view,2026-03-17 07:28:29,Tablet,Google,P014,2026-03,2026-03
2,E00008623,U01608,U01608_S03,app_open,2026-03-25 13:09:00,Mobile,YouTube,NaN,2026-03,2026-03
3,E00001776,U00315,U00315_S02,app_open,2026-02-13 09:22:00,Mobile,Direct,NaN,2026-02,2026-02
4,E00003466,U00631,U00631_S03,product_view,2026-02-03 11:07:20,Mobile,Google,P004,2026-02,2026-01


In [41]:
#count unique users in each cohort and activity month
cohort_data = df_cohort.groupby(["cohort_month","activity_month"])["user_id"].nunique().reset_index()

In [42]:
cohort_data

,cohort_month,activity_month,user_id
0,2026-01,2026-01,668
1,2026-01,2026-02,160
2,2026-01,2026-03,109
3,2026-02,2026-02,659
4,2026-02,2026-03,186
5,2026-03,2026-03,673


In [46]:
cohort_data["cohort_index"] = (
    (cohort_data["activity_month"].dt.year - cohort_data["cohort_month"].dt.year) * 12 +
    (cohort_data["activity_month"].dt.month - cohort_data["cohort_month"].dt.month)
)

In [47]:
cohort_data.head()

,cohort_month,activity_month,user_id,cohort_index
0,2026-01,2026-01,668,0
1,2026-01,2026-02,160,1
2,2026-01,2026-03,109,2
3,2026-02,2026-02,659,0
4,2026-02,2026-03,186,1


In [49]:
cohort_pivot = cohort_data.pivot(index="cohort_month",
                                 columns="cohort_index",
                                                                  values="user_id")


In [50]:
cohort_pivot

cohort_index,0,1,2
cohort_month,,,
2026-01,668.0,160.0,109.0
2026-02,659.0,186.0,NaN
2026-03,673.0,NaN,NaN


In [51]:
#calculate cohort retention percentage
cohort_size = cohort_pivot.iloc[:,0]
retention= cohort_pivot.divide(cohort_size, axis=0) * 100

In [52]:
retention

cohort_index,0,1,2
cohort_month,,,
2026-01,100.0,23.952096,16.317365
2026-02,100.0,28.224583,NaN
2026-03,100.0,NaN,NaN


In [53]:
user_sessions = df_clean.groupby("user_id")["session_id"].nunique()

In [54]:
user_sessions

,session_id
user_id,
U00001,3
U00002,3
U00003,2
U00004,2
U00005,1
...,...
U01996,1
U01997,1
U01998,2


In [55]:
repeat_users = (user_sessions>1).sum()

In [56]:
repeat_users

np.int64(1168)

In [57]:
repeat_engagement_rate = (repeat_users / user_sessions.count()) * 100

In [58]:
repeat_engagement_rate

np.float64(58.4)

In [59]:
device_funnel = df_clean.groupby(["device","event_name"])["user_id"].nunique().unstack(fill_value=0)
device_funnel = device_funnel.reindex(columns=["app_open","product_view","add_to_cart","checkout",
                                               "purchase"],
                                                                                     fill_value=0)


In [60]:
device_funnel

event_name,app_open,product_view,add_to_cart,checkout,purchase
device,,,,,
Desktop,475,475,289,165,90
Mobile,1425,1425,819,526,321
Tablet,100,100,51,33,23


In [62]:
source_funnel = df_clean.groupby(["acquisition_source","event_name"])["user_id"].nunique()

In [63]:
source_funnel = source_funnel.unstack(fill_value=0)

In [64]:
source_funnel

event_name,add_to_cart,app_open,checkout,product_view,purchase
acquisition_source,,,,,
Direct,200,357,118,357,69
Email,88,158,56,158,39
Google,356,613,233,613,138
Instagram,276,466,163,466,95
Referral,110,186,73,186,41
YouTube,129,220,81,220,52


In [65]:
cohort_funnel = df_cohort.groupby(["cohort_month","event_name"])["user_id"].nunique().unstack(fill_value=0)

cohort_funnel

event_name,add_to_cart,app_open,checkout,product_view,purchase
cohort_month,,,,,
2026-01,384,668,233,668,132
2026-02,382,659,233,659,146
2026-03,393,673,258,673,156


In [66]:
df_sorted["time_diff"] = df_sorted.groupby("user_id")["timestamp"].diff()

In [67]:
df_sorted[["user_id","timestamp","time_diff"]].head()


,user_id,timestamp,time_diff
1478,U00001,2026-01-09 19:39:00,NaT
9185,U00001,2026-01-09 19:39:45,0 days 00:00:45
6344,U00001,2026-01-09 19:41:29,0 days 00:01:44
516,U00001,2026-01-23 20:27:00,14 days 00:45:31
10386,U00001,2026-01-23 20:27:55,0 days 00:00:55


In [68]:
df_sorted["new_session_30"] = df_sorted["time_diff"].isna()
(df_sorted["time_diff"]> pd.Timedelta(minutes=30))

,time_diff
1478,False
9185,False
6344,False
516,True
10386,False
...,...
5613,False
1138,False
3837,True
8043,False


In [69]:
df_sorted["new_session_30"].value_counts()

,count
new_session_30,
False,8674
True,2000


In [70]:
df_sorted["new_session_60"] = df_sorted["time_diff"].isna()
(df_sorted["time_diff"]> pd.Timedelta(minutes=60))

,time_diff
1478,False
9185,False
6344,False
516,True
10386,False
...,...
5613,False
1138,False
3837,True
8043,False


In [71]:
df_sorted["new_session_60"].value_counts()

,count
new_session_60,
False,8674
True,2000


In [72]:
df_sorted[["new_session_30","new_session_60"]].sum()

,0
new_session_30,2000
new_session_60,2000


In [73]:
#Funnel 1: App Open -> product view -> purchase
#Funnel 2: App open -> add to cart -> purchase

In [74]:
#observational findings: the number of users decreases at later funnel stages.




 #recommended experiment: text weather simplifying checkout improves purchase conversions.

 #causal result: a control experiment is needed to confirm causality.